# B04 · S6 — Planificación, percepción y programación

**Objetivo (RA4-b/c):** ver a un robot **decidir con una cámara** y comparar dos técnicas de programación (reglas vs. lógica difusa) sobre el mismo problema: seguir una línea.

> Práctica guiada de la S6 de los [apuntes](../apuntes.md). Simulación con `aitk.robots`; la pista se dibuja en el propio notebook.

In [ ]:
%pip install aitk aitk.robots pillow scikit-fuzzy networkx

## 1. La pista y el robot

Un Scribbler con cámara (64×32) sobre una pista circular dibujada con PIL. La cámara ve en gris la línea negra sobre fondo blanco.

In [ ]:
import numpy as np
from PIL import Image, ImageDraw
import aitk.robots as bots

img = Image.new("RGB", (220, 180), "white")
ImageDraw.Draw(img).ellipse([50, 30, 170, 150], outline="black", width=14)
img.save("pista.png")

world = bots.World(220, 180, boundary_wall_color="yellow", ground_image_filename="pista.png")
robot = bots.Scribbler(x=105, y=95, a=90)
robot.add_device(bots.Camera(64, 32))
world.add_robot(robot)
print("mundo listo")

## 2. Controlador por REGLAS (todo lo decides tú)

1. Toma la imagen de la cámara.
2. Encuentra los **píxeles oscuros** (la línea).
3. Calcula el **centroide x** y su desvío respecto al centro de la imagen.
4. Avanza y corrige proporcionalmente al desvío.

In [ ]:
def seguir_por_reglas(robot):
    a = np.asarray(robot["camera"].get_image())
    oscuros = np.where(a.mean(axis=2) < 100)
    if len(oscuros[0]) == 0:
        robot.move(0.2, 0.3)                         # no ve linea: busca girando
        return
    desvio = (oscuros[1].mean() / a.shape[1]) - 0.5  # centroide x normalizado
    robot.move(0.5, desvio * 1.5)                    # avanza y corrige

world.reset()
world.seconds(8, [seguir_por_reglas], real_time=False)
world.display()   # imagen final del mundo
# world.watch()   # descomenta para ver el video completo

## 3. Actividad A2 · variante DIFUSA (entregable)

Sustituye la corrección proporcional por un **sistema difuso**: el desvío se fuzzifica en `izquierda / centro / derecha` y la salida es el giro. Compara ambos controladores (oscilación, código, explicabilidad).

La estructura ya está montada; completa las reglas y ejecuta.

In [ ]:
import skfuzzy as fuzz
from skfuzzy import control as ctrl

desvio = ctrl.Antecedent(np.arange(-0.5, 0.51, 0.01), "desvio")
giro = ctrl.Consequent(np.arange(-1.5, 1.51, 0.01), "giro")

desvio["izq"] = fuzz.trimf(desvio.universe, [-0.5, -0.5, 0])
desvio["centro"] = fuzz.trimf(desvio.universe, [-0.25, 0, 0.25])
desvio["der"] = fuzz.trimf(desvio.universe, [0, 0.5, 0.5])
giro["izq"] = fuzz.trimf(giro.universe, [-1.5, -1.5, 0])
giro["centro"] = fuzz.trimf(giro.universe, [-0.5, 0, 0.5])
giro["der"] = fuzz.trimf(giro.universe, [0, 1.5, 1.5])

# TODO A2: define las 3 reglas difusas y completa el controlador
reglas = [
    # ctrl.Rule(desvio["izq"], giro["izq"]),
    # ...
]
sim = ctrl.ControlSystemSimulation(ctrl.ControlSystem(reglas))

def seguir_difuso(robot):
    a = np.asarray(robot["camera"].get_image())
    osc = np.where(a.mean(axis=2) < 100)
    if len(osc[0]) == 0:
        robot.move(0.2, 0.3); return
    d = (osc[1].mean() / a.shape[1]) - 0.5
    sim.input["desvio"] = float(np.clip(d, -0.5, 0.5))
    sim.compute()
    robot.move(0.5, float(sim.output["giro"]))

world.reset()
world.seconds(8, [seguir_difuso], real_time=False)
world.display()

## 4. Cierre: ¿qué gana y qué pierde cada técnica?

| | Reglas | Lógica difusa |
|---|---|---|
| Código | Todo a mano | Variables + reglas + motor |
| Suavidad | Saltos en el giro | Transición continua |
| Explicabilidad | Total | Alta (reglas legibles) |
| Datos | Ninguno | Ninguno |

Es la misma tensión interpretabilidad/potencia que viste en RA5: las reglas se leen; una red neuronal (el siguiente paso de la UD04) no.

**Para casa:** responde en 3 líneas por qué la **odometría** se degrada sin límite y cómo lo corrige el **filtro de partículas**.